# step1_ref_faces — 참조 얼굴 풀 구축 (조합 3용)

GPT-Image-2로 가상 얼굴 20장을 생성한다.
InstantID가 얼굴 임베딩만 사용하므로 머리카락 없는 삭발 이미지로 만든다.
먼저 파일럿 4장으로 얼굴이 실제로 구분되는지 확인한 뒤 나머지를 생성한다.

In [ ]:
!pip install -q openai

from getpass import getpass
from openai import OpenAI
import os, base64, time
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/saloncut_data"
REF_DIR = f"{BASE}/ref_faces"
os.makedirs(REF_DIR, exist_ok=True)

client = OpenAI(api_key=getpass("OpenAI API key: "))
print("준비 완료:", REF_DIR)

## 파일럿 4장

프롬프트로 얼굴이 실제로 구분되는지 먼저 확인한다.
얼굴형·눈·코·입 네 항목을 서로 겹치지 않게 배치했다.

In [ ]:
MODEL = "gpt-image-2"

SUFFIX = (
    "completely bald with no hair on the head, eyebrows present, "
    "facing directly forward, eyes open looking at camera, neutral relaxed expression, "
    "head and shoulders only, plain black top, "
    "solid light gray background, even soft studio lighting, minimal shadows, "
    "no makeup, no accessories, no glasses, "
    "photorealistic portrait photograph, sharp focus on the face"
)

PILOT = {
    "ref-01": "a Korean woman in her 20s, round face shape with full cheeks, "
              "monolid eyes with no visible crease, small narrow eyes, "
              "low flat nose bridge with rounded nose tip, thin lips",
    "ref-02": "a Korean woman in her 20s, slim oval face shape with narrow jaw, "
              "large wide double-lidded eyes with deep crease, "
              "high straight nose bridge with defined nose tip, full lips",
    "ref-03": "a Korean woman in her 30s, angular face shape with prominent cheekbones, "
              "long narrow eyes with upturned outer corners, "
              "straight nose bridge with slightly pointed tip, medium lips",
    "ref-04": "a Korean man in his 20s, long face shape with square jaw, "
              "thick heavy eyelids over small eyes, "
              "wide nose with broad nostrils, thin firm lips",
}

pilot_imgs = {}
for rid, desc in PILOT.items():
    t0 = time.time()
    r = client.images.generate(
        model=MODEL, prompt=f"{desc}, {SUFFIX}",
        size="1024x1024", n=1,
    )
    img = Image.open(BytesIO(base64.b64decode(r.data[0].b64_json)))
    img.save(f"{REF_DIR}/{rid}.png")
    pilot_imgs[rid] = img
    print(f"{rid}  {round(time.time()-t0,1)}초")

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, (rid, img) in zip(axes, pilot_imgs.items()):
    ax.imshow(img); ax.set_title(rid, fontsize=14); ax.axis("off")
plt.tight_layout()
plt.show()

## 파일럿 검증 — InsightFace 검출·임베딩

생성한 얼굴이 InstantID에서 쓸 수 있는지 확인한다.
검출 여부와 얼굴 크기, 그리고 4장의 임베딩이 서로 구분되는지 본다.

In [ ]:
import numpy as np, cv2, os, torch, glob, shutil
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/saloncut_data"
REF_DIR = f"{BASE}/ref_faces"

print("numpy", np.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("ref_faces:", sorted(os.listdir(REF_DIR)))

In [ ]:
from insightface.app import FaceAnalysis

# antelopev2 압축 해제 시 폴더가 한 겹 더 생기는 문제 정리
p = "/content/models/antelopev2"
if os.path.isdir(f"{p}/antelopev2"):
    for f in os.listdir(f"{p}/antelopev2"):
        shutil.move(f"{p}/antelopev2/{f}", f"{p}/{f}")
    os.rmdir(f"{p}/antelopev2")

app = FaceAnalysis(name='antelopev2', root='/content',
                   providers=['CPUExecutionProvider'])
app.prepare(ctx_id=-1, det_size=(640, 640))

RIDS = ["ref-01", "ref-02", "ref-03", "ref-04"]
embs = {}
for rid in RIDS:
    img = Image.open(f"{REF_DIR}/{rid}.png").convert("RGB")
    faces = app.get(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    if not faces:
        print(f"{rid}  검출 실패"); continue
    f = max(faces, key=lambda x: (x.bbox[2]-x.bbox[0]) * (x.bbox[3]-x.bbox[1]))
    embs[rid] = f.normed_embedding
    print(f"{rid}  얼굴 {int(f.bbox[2]-f.bbox[0])}x{int(f.bbox[3]-f.bbox[1])}  det_score {f.det_score:.3f}")

print("\n=== 임베딩 상호 유사도 ===")
ids = list(embs)
print("      " + "  ".join(f"{i:>7}" for i in ids))
for a in ids:
    print(f"{a:>6}  " + "  ".join(f"{np.dot(embs[a], embs[b]):7.3f}" for b in ids))

## 조합 3 로딩

InstantID + RealVisXL 조합. img2img 파이프라인을 쓴다.
IP-Adapter 가중치와 ControlNet을 먼저 받는다.

In [ ]:
from huggingface_hub import hf_hub_download

for f in ["ControlNetModel/config.json",
          "ControlNetModel/diffusion_pytorch_model.safetensors",
          "ip-adapter.bin"]:
    hf_hub_download(repo_id="InstantX/InstantID", filename=f,
                    local_dir="/content/checkpoints")

!ls -R /content/checkpoints

In [ ]:
import sys, time
sys.path.append("/content/InstantID")

from diffusers.models import ControlNetModel
from pipeline_stable_diffusion_xl_instantid_img2img import StableDiffusionXLInstantIDImg2ImgPipeline

t0 = time.time()
controlnet = ControlNetModel.from_pretrained(
    "/content/checkpoints/ControlNetModel", torch_dtype=torch.float16)

pipe3 = StableDiffusionXLInstantIDImg2ImgPipeline.from_pretrained(
    "SG161222/RealVisXL_V4.0", controlnet=controlnet,
    torch_dtype=torch.float16, use_safetensors=True, variant="fp16",
).to("cuda")

pipe3.load_ip_adapter_instantid("/content/checkpoints/ip-adapter.bin")

print(f"로딩 {round(time.time()-t0,1)}초")
print(f"VRAM {torch.cuda.memory_allocated()/1024**3:.2f}GB")

## 파일럿 검증 — 조합 3 적용

참조 얼굴 4종을 같은 손님 사진에 적용해 결과가 구분되는지 확인한다.
공통 스펙(정면·균일 조명·삭발)이 실제 손님 사진과 맞는지도 함께 본다.

In [ ]:
import matplotlib.pyplot as plt
from insightface.utils import face_align

TEST_NORMAL = f"{BASE}/test_images/normal"
TARGET = "normal_01_short_dark"

face_img = Image.open(f"{TEST_NORMAL}/{TARGET}.jpg").convert("RGB")
w, h = face_img.size
scale = 1024 / max(w, h)
face_img = face_img.resize(((int(w*scale)//8)*8, (int(h*scale)//8)*8))

# 손님 사진에서 랜드마크(포즈 제어용) 추출
tf = app.get(cv2.cvtColor(np.array(face_img), cv2.COLOR_RGB2BGR))
tf = sorted(tf, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]))[-1]
kps = tf.kps
print("손님 사진 랜드마크 추출 완료", face_img.size)

PROMPT = "photorealistic portrait, natural skin texture, soft studio lighting"
NEG = "blurry, distorted, deformed face, extra features, cartoon, watermark"

results = {}
for rid in RIDS:
    t0 = time.time()
    out = pipe3(
        prompt=PROMPT, negative_prompt=NEG,
        image_embeds=embs[rid],
        image=face_img,
        control_image=face_img,
        controlnet_conditioning_scale=0.8,
        ip_adapter_scale=0.8,
        strength=0.8,
        num_inference_steps=30,
        guidance_scale=5.0,
    ).images[0]
    results[rid] = out
    print(f"{rid}  {round(time.time()-t0,1)}초")

fig, axes = plt.subplots(1, 5, figsize=(25, 7))
axes[0].imshow(face_img); axes[0].set_title("원본", fontsize=14)
for ax, rid in zip(axes[1:], RIDS):
    ax.imshow(results[rid]); ax.set_title(rid, fontsize=14)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import sys, time
sys.path.insert(0, "/content/InstantID")

del pipe3
import gc; gc.collect(); torch.cuda.empty_cache()

from diffusers.models import ControlNetModel
from pipeline_stable_diffusion_xl_instantid_img2img import StableDiffusionXLInstantIDImg2ImgPipeline
from pipeline_stable_diffusion_xl_instantid import draw_kps

t0 = time.time()
controlnet = ControlNetModel.from_pretrained(
    "/content/checkpoints/ControlNetModel", torch_dtype=torch.float16)
pipe3 = StableDiffusionXLInstantIDImg2ImgPipeline.from_pretrained(
    "SG161222/RealVisXL_V5.0", controlnet=controlnet, torch_dtype=torch.float16,
).to("cuda")
pipe3.load_ip_adapter_instantid("/content/checkpoints/ip-adapter.bin")
print(f"로딩 {round(time.time()-t0,1)}초  VRAM {torch.cuda.memory_allocated()/1024**3:.2f}GB")


def prep_instantid(ref_path, src_path):
    """참조에서 임베딩, 원본에서 키포인트 이미지를 만든다."""
    ref_faces = app.get(cv2.imread(ref_path))
    if not ref_faces:
        raise ValueError("참조 얼굴 검출 실패")
    emb = ref_faces[0]['embedding']

    src_faces = app.get(cv2.imread(src_path))
    if not src_faces:
        raise ValueError("원본 얼굴 검출 실패")
    face = sorted(src_faces, key=lambda x: x['bbox'][2] * x['bbox'][3])[-1]

    src_pil = Image.open(src_path).convert("RGB")
    return emb, draw_kps(src_pil, face['kps']), src_pil

print("준비 완료")

In [ ]:
!pip install -q mediapipe
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print("설치 완료")

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageFilter
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from skimage.metrics import structural_similarity as ssim


landmarker = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path="/content/face_landmarker.task"),
        num_faces=1))

def build_face_mask(img):
    w, h = img.size
    r = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(img)))
    lm = r.face_landmarks[0]
    oval = vision.FaceLandmarksConnections.FACE_LANDMARKS_FACE_OVAL
    pts = np.array([[int(lm[c.start].x*w), int(lm[c.start].y*h)] for c in oval])
    m = np.zeros((h, w), np.uint8); cv2.fillPoly(m, [pts], 255)
    return Image.fromarray(m)

def recompose_soft(original, result, mask, blur=25):
    size = result.size
    return Image.composite(result, original.resize(size),
                           mask.resize(size).filter(ImageFilter.GaussianBlur(blur)))

def mask_outside_ssim(original, result, mask):
    size = result.size
    a = np.array(original.resize(size).convert("L")); b = np.array(result.convert("L"))
    m = np.array(mask.resize(size)) < 128
    _, s = ssim(a, b, full=True)
    return float(s[m].mean())


TEST_NORMAL = f"{BASE}/test_images/normal"
SRC = f"{TEST_NORMAL}/normal_01_short_dark.jpg"

results, comps = {}, {}
for rid in RIDS:
    emb, kps_img, src_pil = prep_instantid(f"{REF_DIR}/{rid}.png", SRC)
    w, h = src_pil.size
    s = 1024 / max(w, h); nw, nh = (int(w*s)//8)*8, (int(h*s)//8)*8
    src_r, kps_r = src_pil.resize((nw, nh)), kps_img.resize((nw, nh))

    t0 = time.time()
    out = pipe3(
        prompt="a person in a hair salon, natural lighting, photorealistic",
        negative_prompt="blurry, low quality, deformed, watermark, text",
        image_embeds=emb, image=src_r, control_image=kps_r,
        strength=0.4, controlnet_conditioning_scale=0.8, ip_adapter_scale=0.8,
        num_inference_steps=30, guidance_scale=5.0,
        generator=torch.Generator("cuda").manual_seed(42),
    ).images[0]
    mask = build_face_mask(src_pil)
    comp = recompose_soft(src_pil, out, mask, 25)
    results[rid], comps[rid] = out, comp
    print(f"{rid}  {round(time.time()-t0,1)}초  마스크밖 SSIM {mask_outside_ssim(src_pil, comp, mask):.4f}")

fig, axes = plt.subplots(1, 5, figsize=(25, 7))
axes[0].imshow(src_pil); axes[0].set_title("original")
for ax, rid in zip(axes[1:], RIDS):
    ax.imshow(comps[rid]); ax.set_title(rid)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
def identity_similarity(a, b):
    """두 이미지의 최대 얼굴 임베딩 코사인 유사도. 높을수록 같은 인물."""
    def emb(im):
        fs = app.get(cv2.cvtColor(np.array(im.convert("RGB")), cv2.COLOR_RGB2BGR))
        return None if not fs else max(fs, key=lambda x: x.bbox[2]-x.bbox[0]).normed_embedding
    ea, eb = emb(a), emb(b)
    return float("nan") if ea is None or eb is None else float(np.dot(ea, eb))

print("준비 완료")

In [ ]:
def get_kps(img_rgb):
    fs = app.get(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return None if not fs else max(fs, key=lambda f: f.bbox[2]-f.bbox[0]).kps.astype(np.float32)

def align_then_recompose(original, result, mask, blur=25):
    """생성 결과를 원본 좌표계로 정렬한 뒤 재합성."""
    res = result.resize(original.size)
    o, g = np.array(original), np.array(res)
    ko, kg = get_kps(o), get_kps(g)
    if ko is None or kg is None:
        return recompose_soft(original, res, mask, blur)
    M, _ = cv2.estimateAffinePartial2D(kg, ko, method=cv2.LMEDS)
    warped = cv2.warpAffine(g, M, (o.shape[1], o.shape[0]), flags=cv2.INTER_LANCZOS4)
    return recompose_soft(original, Image.fromarray(warped), mask, blur)


mask = build_face_mask(src_pil)
aligned = {}
for rid in RIDS:
    aligned[rid] = align_then_recompose(src_pil, results[rid], mask, 25)
    print(f"{rid}  마스크밖 SSIM {mask_outside_ssim(src_pil, aligned[rid], mask):.4f}  "
          f"원본↔결과 {identity_similarity(src_pil, aligned[rid]):.4f}")

fig, axes = plt.subplots(2, 5, figsize=(25, 14))
axes[0][0].imshow(src_pil); axes[0][0].set_title("original")
axes[1][0].imshow(src_pil); axes[1][0].set_title("original")
for i, rid in enumerate(RIDS, start=1):
    axes[0][i].imshow(comps[rid]);   axes[0][i].set_title(f"{rid} 정렬 없음")
    axes[1][i].imshow(aligned[rid]); axes[1][i].set_title(f"{rid} 정렬 적용")
for row in axes:
    for ax in row: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
seg_url = "https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite"
!wget -q -O /content/selfie_multiclass.tflite {seg_url}

segmenter = vision.ImageSegmenter.create_from_options(
    vision.ImageSegmenterOptions(
        base_options=mp_python.BaseOptions(model_asset_path="/content/selfie_multiclass.tflite"),
        output_category_mask=True))

def get_hair_mask_dilated(img, px=20):
    """머리카락 클래스(1)를 추출해 px만큼 팽창."""
    cat = np.squeeze(segmenter.segment(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(img))).category_mask.numpy_view())
    hair = ((cat == 1) * 255).astype(np.uint8)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px, px))
    return Image.fromarray(cv2.dilate(hair, k))

def recompose_with_hair(original, result, face_mask, hair_mask, blur=25, hair_blur=3):
    """얼굴 경계는 부드럽게, 머리카락은 원본으로 복원."""
    size = original.size
    res = result.resize(size)
    comp = Image.composite(res, original,
                           face_mask.resize(size).filter(ImageFilter.GaussianBlur(blur)))
    return Image.composite(original, comp,
                           hair_mask.resize(size).filter(ImageFilter.GaussianBlur(hair_blur)))

def align_then_recompose_hair(original, result, face_mask, hair_mask, blur=25, hair_blur=3):
    """kps 정렬 후 헤어 복원 재합성."""
    res = result.resize(original.size)
    o, g = np.array(original), np.array(res)
    ko, kg = get_kps(o), get_kps(g)
    if ko is not None and kg is not None:
        M, _ = cv2.estimateAffinePartial2D(kg, ko, method=cv2.LMEDS)
        res = Image.fromarray(cv2.warpAffine(g, M, (o.shape[1], o.shape[0]),
                                             flags=cv2.INTER_LANCZOS4))
    return recompose_with_hair(original, res, face_mask, hair_mask, blur, hair_blur)

def fringe_region_ssim(original, result, hair_mask, face_mask):
    """헤어 ∩ 얼굴 영역의 SSIM. 이마를 덮은 앞머리를 측정."""
    a = np.array(original.convert("L"))
    b = np.array(result.resize(original.size).convert("L"))
    h = np.array(hair_mask.resize(original.size)) > 127
    f = np.array(face_mask.resize(original.size)) > 127
    region = h & f
    if region.sum() < 100:
        return float("nan")
    _, smap = ssim(a, b, full=True)
    return float(smap[region].mean())

print("함수 준비 완료")

In [ ]:
hair = get_hair_mask_dilated(src_pil, 20)

d1 = {}
for rid in RIDS:
    d1[rid] = align_then_recompose_hair(src_pil, results[rid], mask, hair)
    print(f"{rid}  앞머리 SSIM {fringe_region_ssim(src_pil, d1[rid], hair, mask):.4f} "
          f"(정렬만 {fringe_region_ssim(src_pil, aligned[rid], hair, mask):.4f})  "
          f"원본↔결과 {identity_similarity(src_pil, d1[rid]):.4f}")

fig, axes = plt.subplots(2, 5, figsize=(25, 14))
for r, (label, d) in enumerate([("정렬만", aligned), ("정렬+D-1", d1)]):
    axes[r][0].imshow(src_pil); axes[r][0].set_title("original")
    for i, rid in enumerate(RIDS, start=1):
        axes[r][i].imshow(d[rid]); axes[r][i].set_title(f"{rid} {label}")
    for ax in axes[r]: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
!apt-get install -qq -y fonts-nanum > /dev/null
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print("폰트 완료")

In [ ]:
def get_neck_mask(img, face_mask):
    """목·데콜테 영역. body-skin(클래스 2) 중 얼굴 마스크 아래쪽."""
    cat = np.squeeze(segmenter.segment(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(img))).category_mask.numpy_view())
    body = (cat == 2)
    fm_ = np.array(face_mask.resize(img.size)) > 127
    ys = np.where(fm_.any(axis=1))[0]
    if len(ys) == 0: return None
    neck = np.zeros_like(body); neck[ys.max():, :] = body[ys.max():, :]
    return neck

def face_neck_delta_e(img, face_mask, hair_mask=None):
    """얼굴 평균색과 목 평균색의 CIE ΔE."""
    lab = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2LAB).astype(np.float64)
    lab[..., 0] *= 100/255; lab[..., 1:] -= 128
    fm_ = np.array(face_mask.resize(img.size)) > 127
    if hair_mask is not None:
        fm_ &= ~(np.array(hair_mask.resize(img.size)) > 127)
    neck = get_neck_mask(img, face_mask)
    if neck is None or neck.sum() < 500 or fm_.sum() < 500: return None
    return float(np.sqrt(((lab[fm_].mean(axis=0) - lab[neck].mean(axis=0))**2).sum()))

def face_stats(img, face_mask, hair_mask=None):
    """얼굴 영역의 밝기 평균·표준편차, 고주파 성분(질감)."""
    arr = np.array(img.convert("RGB"))
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float64)
    L = lab[..., 0] * 100/255
    fm_ = np.array(face_mask.resize(img.size)) > 127
    if hair_mask is not None:
        fm_ &= ~(np.array(hair_mask.resize(img.size)) > 127)
    lap = cv2.Laplacian(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY), cv2.CV_64F)
    return {"L 평균": round(L[fm_].mean(), 2), "L 표준편차": round(L[fm_].std(), 2),
            "질감": round(lap[fm_].var(), 1)}

print("지표 함수 준비 완료")

In [ ]:
import pandas as pd

rows = []
for name, img in [("원본", src_pil)] + [(rid, d1[rid]) for rid in RIDS]:
    st = face_stats(img, mask, hair)
    de = face_neck_delta_e(img, mask, hair)
    rows.append({
        "조건": name,
        "질감": st["질감"],
        "ΔE": round(de, 2) if de is not None else None,
        "L 평균": st["L 평균"],
        "L 표준편차": st["L 표준편차"],
        "원본↔결과": round(identity_similarity(src_pil, img), 4) if name != "원본" else "—",
    })

df_c3 = pd.DataFrame(rows)
print(df_c3.to_string(index=False))
print("\n참고 — 조합 5(같은 사진): 원본 질감 63.4 / 결과 10.9, 원본 ΔE 8.81 / 결과 10.17")

In [ ]:
def reinhard_transfer_a(src, ref, m, alpha=1.0):
    """src의 마스크 영역 색 통계를 ref 쪽으로 alpha만큼 이동."""
    s = cv2.cvtColor(np.array(src.convert("RGB")), cv2.COLOR_RGB2LAB).astype(np.float32)
    r = cv2.cvtColor(np.array(ref.resize(src.size).convert("RGB")), cv2.COLOR_RGB2LAB).astype(np.float32)
    mk = np.array(m.resize(src.size)) > 127
    out = s.copy()
    for c in range(3):
        ss, sm = s[..., c][mk].std(), s[..., c][mk].mean()
        rs, rm = r[..., c][mk].std(), r[..., c][mk].mean()
        ts, tm = ss + (rs - ss) * alpha, sm + (rm - sm) * alpha
        if ss > 1e-6:
            out[..., c] = (s[..., c] - sm) / ss * ts + tm
    return Image.fromarray(cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB))

def transfer_high_freq(base, original, m, strength=0.5, radius=3):
    """원본의 고주파 성분을 base의 마스크 영역에 더한다."""
    b = np.array(base.convert("RGB")).astype(np.float32)
    o = np.array(original.resize(base.size).convert("RGB")).astype(np.float32)
    hf = o - cv2.GaussianBlur(o, (0, 0), radius)
    mk = (np.array(m.resize(base.size)) > 127).astype(np.float32)[..., None]
    return Image.fromarray(np.clip(b + hf * mk * strength, 0, 255).astype(np.uint8))


RID = "ref-01"

# 생성용 마스크 = FACE_OVAL - 헤어 (고주파 이식 대상 영역)
fa = np.array(mask).copy()
fa[np.array(hair) > 127] = 0
gen_mask = Image.fromarray(fa)

A = d1[RID]
raw_B = reinhard_transfer_a(results[RID], src_pil, gen_mask, 1.0)
B = align_then_recompose_hair(src_pil, raw_B, mask, hair)

conds = {"A 현재(D-1)": A, "B 색정합": B}
for s in [0.3, 0.5, 0.7]:
    conds[f"A+고주파 {s}"] = transfer_high_freq(A, src_pil, gen_mask, s)
    conds[f"B+고주파 {s}"] = transfer_high_freq(B, src_pil, gen_mask, s)

rows = []
for name, img in [("원본", src_pil)] + list(conds.items()):
    st = face_stats(img, mask, hair)
    de = face_neck_delta_e(img, mask, hair)
    rows.append({
        "조건": name, "질감": st["질감"],
        "ΔE": round(de, 2) if de is not None else None,
        "L 평균": st["L 평균"], "L 표준편차": st["L 표준편차"],
        "원본↔결과": round(identity_similarity(src_pil, img), 4) if name != "원본" else "—",
    })
print(pd.DataFrame(rows).to_string(index=False))

show = ["A 현재(D-1)", "B 색정합", "A+고주파 0.5", "B+고주파 0.5"]
fig, axes = plt.subplots(1, 5, figsize=(25, 7))
axes[0].imshow(src_pil); axes[0].set_title("원본", fontsize=13)
for ax, k in zip(axes[1:], show):
    ax.imshow(conds[k]); ax.set_title(k, fontsize=13)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
SRC9 = f"{TEST_NORMAL}/normal_09_dark_skin.jpg"
RID9 = "ref-04"

emb9, kps9, src9 = prep_instantid(f"{REF_DIR}/{RID9}.png", SRC9)
w, h = src9.size
s = 1024 / max(w, h); nw, nh = (int(w*s)//8)*8, (int(h*s)//8)*8

t0 = time.time()
out9 = pipe3(
    prompt="a person in a hair salon, natural lighting, photorealistic",
    negative_prompt="blurry, low quality, deformed, watermark, text",
    image_embeds=emb9, image=src9.resize((nw, nh)), control_image=kps9.resize((nw, nh)),
    strength=0.4, controlnet_conditioning_scale=0.8, ip_adapter_scale=0.8,
    num_inference_steps=30, guidance_scale=5.0,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]
print(f"생성 {round(time.time()-t0,1)}초")

mask9 = build_face_mask(src9)
hair9 = get_hair_mask_dilated(src9, 20)
fa9 = np.array(mask9).copy(); fa9[np.array(hair9) > 127] = 0
gen_mask9 = Image.fromarray(fa9)

A9 = align_then_recompose_hair(src9, out9, mask9, hair9)
B9 = align_then_recompose_hair(src9, reinhard_transfer_a(out9, src9, gen_mask9, 1.0), mask9, hair9)

conds9 = {"A 현재(D-1)": A9, "B 색정합": B9}
for st_ in [0.3, 0.5, 0.7]:
    conds9[f"A+고주파 {st_}"] = transfer_high_freq(A9, src9, gen_mask9, st_)
    conds9[f"B+고주파 {st_}"] = transfer_high_freq(B9, src9, gen_mask9, st_)

rows9 = []
for name, img in [("원본", src9)] + list(conds9.items()):
    stt = face_stats(img, mask9, hair9)
    de = face_neck_delta_e(img, mask9, hair9)
    rows9.append({
        "조건": name, "질감": stt["질감"],
        "ΔE": round(de, 2) if de is not None else None,
        "L 평균": stt["L 평균"], "L 표준편차": stt["L 표준편차"],
        "원본↔결과": round(identity_similarity(src9, img), 4) if name != "원본" else "—",
    })
print(pd.DataFrame(rows9).to_string(index=False))

show = ["A 현재(D-1)", "B 색정합", "A+고주파 0.5", "B+고주파 0.5"]
fig, axes = plt.subplots(1, 5, figsize=(25, 8))
axes[0].imshow(src9); axes[0].set_title("원본", fontsize=13)
for ax, k in zip(axes[1:], show):
    ax.imshow(conds9[k]); ax.set_title(k, fontsize=13)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
from getpass import getpass
from openai import OpenAI
import base64
from io import BytesIO

client = OpenAI(api_key=getpass("OpenAI API key: "))
print("준비 완료")

In [ ]:
SUFFIX = (
    "completely bald with no hair on the head, eyebrows present, "
    "facing directly forward, eyes open looking at camera, neutral relaxed expression, "
    "head and shoulders only, plain black top, "
    "solid light gray background, even soft studio lighting, minimal shadows, "
    "no makeup, no accessories, no glasses, "
    "photorealistic portrait photograph, sharp focus on the face"
)

PILOT2 = {
    "test_attractive":
        "a Korean woman in her 20s, attractive and photogenic, "
        "slim oval face with clean jawline, large clear double-lidded eyes, "
        "high straight nose bridge, well-balanced symmetric features, smooth even skin",
    "test_ordinary":
        "a Korean woman in her 20s, an ordinary everyday person you would see on the street, "
        "average unremarkable features, slightly asymmetric face, "
        "natural skin with visible pores, faint blemishes and uneven tone, "
        "not a fashion model, plain-looking but healthy",
}

p2 = {}
for name, desc in PILOT2.items():
    t0 = time.time()
    r = client.images.generate(model="gpt-image-2", prompt=f"{desc}, {SUFFIX}",
                               size="1024x1024", n=1)
    img = Image.open(BytesIO(base64.b64decode(r.data[0].b64_json)))
    img.save(f"{REF_DIR}/{name}.png")
    p2[name] = img
    print(f"{name}  {round(time.time()-t0,1)}초")

fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, (k, v) in zip(axes, p2.items()):
    ax.imshow(v); ax.set_title(k, fontsize=13); ax.axis("off")
plt.tight_layout(); plt.show()

## 참조 얼굴 32장 생성

팀원 조정안 반영 (8/13). 40대 남 매력형 추가, 50대를 2→4장으로 확대.
그룹 단위로 나눠 생성하고 결과를 확인하며 진행한다.

In [ ]:
!pip install -q openai

import os, time, base64
from io import BytesIO
from getpass import getpass
from PIL import Image
from openai import OpenAI
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/saloncut_data"
REF_DIR = f"{BASE}/ref_faces"
os.makedirs(REF_DIR, exist_ok=True)

client = OpenAI(api_key=getpass("OpenAI API key: "))
print("기존 파일:", sorted(os.listdir(REF_DIR)))

In [ ]:
SUFFIX = (
    "completely bald with no hair on the head, eyebrows present, "
    "facing directly forward, eyes open looking at camera, neutral relaxed expression, "
    "head and shoulders only, plain black top, "
    "solid light gray background, even soft studio lighting, minimal shadows, "
    "no makeup, no accessories, no glasses, "
    "photorealistic portrait photograph, sharp focus on the face"
)

GROUPS = {
"A_한국20여": {
 "ref-01": "a Korean woman in her 20s, attractive and photogenic, slim oval face with clean jawline, large clear double-lidded eyes, high straight nose bridge, well-balanced symmetric features, smooth even skin",
 "ref-02": "a Korean woman in her 20s, attractive and photogenic, round soft face with full cheeks, large round eyes with downturned outer corners, short rounded nose, youthful cute impression, smooth even skin",
 "ref-03": "a Korean woman in her 20s, attractive and photogenic, defined striking features, long upturned almond eyes, sharp nose tip, full lips, elegant sophisticated impression, smooth even skin",
 "ref-04": "a Korean woman in her 20s, an ordinary everyday person you would see on the street, average unremarkable features, wide face with rounded jaw, small monolid eyes, slightly asymmetric face, natural skin with visible pores, faint blemishes and uneven tone, not a fashion model, plain-looking but healthy",
},
"B_한국20남": {
 "ref-05": "a Korean man in his 20s, attractive and photogenic, slim face with defined jawline, large double-lidded eyes, high nose bridge, soft gentle impression, clear even skin",
 "ref-06": "a Korean man in his 20s, attractive and photogenic, angular face with strong jawline, narrow sharp eyes, thick straight eyebrows, high prominent nose bridge, masculine impression, clear even skin",
 "ref-07": "a Korean man in his 20s, an ordinary everyday person you would see on the street, average unremarkable features, square face, small eyes with thick eyelids, slightly asymmetric face, natural skin with visible pores, faint blemishes and uneven tone, not a fashion model, plain-looking but healthy",
},
"C_한국30여": {
 "ref-08": "a Korean woman in her 30s, attractive and photogenic, oval face with defined cheekbones, calm narrow eyes, straight nose bridge, refined mature impression, smooth even skin",
 "ref-09": "a Korean woman in her 30s, attractive and photogenic, slim face with narrow chin, large gentle eyes with long lashes, small refined nose, warm approachable impression, smooth even skin",
 "ref-10": "a Korean woman in her 30s, an ordinary everyday person you would see on the street, average unremarkable features, round face, medium-sized eyes with thin crease, slightly asymmetric face, natural skin with visible pores, faint blemishes and uneven tone, not a fashion model, plain-looking but healthy",
},
"D_한국30남": {
 "ref-11": "a Korean man in his 30s, attractive and photogenic, long face with clean jawline, deep-set double-lidded eyes, straight nose bridge, calm confident impression, clear even skin",
 "ref-12": "a Korean man in his 30s, an ordinary everyday person you would see on the street, average unremarkable features, broad face, narrow monolid eyes, wide flat nose, slightly asymmetric face, natural skin with visible pores, faint blemishes and uneven tone, not a fashion model, plain-looking but healthy",
},
"E_한국40": {
 "ref-13": "a Korean woman in her 40s, attractive and well-groomed, oval face with soft jawline, gentle eyes with fine lines at the corners, straight nose bridge, elegant graceful impression, healthy skin with natural texture",
 "ref-14": "a Korean woman in her 40s, an ordinary everyday person you would see on the street, average unremarkable features, round face with soft cheeks, small eyes with drooping lids, visible smile lines, natural skin with visible pores and age spots, not a fashion model, plain-looking but healthy",
 "ref-15": "a Korean man in his 40s, an ordinary everyday person you would see on the street, average unremarkable features, square face with heavy jaw, small narrow eyes, visible forehead lines, natural skin with visible pores and uneven tone, not a fashion model, plain-looking but healthy",
 "ref-16": "a Korean man in his 40s, attractive and well-groomed, long face with clean jawline, deep-set eyes with fine lines at the corners, high nose bridge, composed dignified impression, healthy skin with natural texture",
},
"F_한국50": {
 "ref-17": "a Korean woman in her 50s, well-groomed and dignified, oval face with gentle jawline, calm eyes with fine wrinkles, straight nose bridge, composed elegant impression, healthy mature skin with natural texture",
 "ref-18": "a Korean woman in her 50s, an ordinary everyday person you would see on the street, average unremarkable features, round face with sagging cheeks, small drooping eyes, deep smile lines, natural skin with visible pores and age spots, not a fashion model, plain-looking but healthy",
 "ref-19": "a Korean man in his 50s, well-groomed and dignified, angular face with firm jawline, calm deep-set eyes with fine wrinkles, straight nose bridge, graying eyebrows, composed refined impression, healthy mature skin with natural texture",
 "ref-20": "a Korean man in his 50s, an ordinary everyday person you would see on the street, average unremarkable features, broad face with sagging cheeks, small drooping eyes, deep forehead and smile lines, graying eyebrows, natural skin with visible pores and age spots, not a fashion model, plain-looking but healthy",
},
"G_외국여": {
 "ref-21": "a Japanese woman in her 20s, attractive and photogenic, small oval face, large round eyes with soft downturned corners, small delicate nose, well-balanced features, smooth even skin",
 "ref-22": "a Chinese woman in her 20s, attractive and photogenic, long oval face, long narrow eyes with upturned corners, high straight nose bridge, well-balanced features, smooth even skin",
 "ref-23": "a Caucasian woman in her 20s, attractive and photogenic, oval face with defined jawline, large deep-set blue eyes, high narrow nose bridge, full lips, well-balanced features, fair smooth skin",
 "ref-24": "a Southeast Asian woman in her 20s, attractive and photogenic, round soft face, large expressive eyes with thick lashes, wide nose with rounded tip, full lips, warm brown skin, well-balanced features",
 "ref-25": "a Black woman in her 20s, attractive and photogenic, oval face with high cheekbones, large almond eyes, broad nose with rounded tip, full lips, rich dark brown skin, well-balanced features",
 "ref-26": "a Middle Eastern woman in her 20s, attractive and photogenic, oval face, large dark eyes with thick lashes and defined brows, high straight nose bridge, olive-toned skin, well-balanced features",
},
"H_외국남": {
 "ref-27": "a Japanese man in his 20s, attractive and photogenic, slim face with soft jawline, narrow gentle eyes, straight nose bridge, well-balanced features, clear even skin",
 "ref-28": "a Chinese man in his 20s, attractive and photogenic, angular face with defined cheekbones, narrow eyes, high nose bridge, well-balanced features, clear even skin",
 "ref-29": "a Caucasian man in his 20s, attractive and photogenic, angular face with strong jawline, deep-set eyes, high prominent nose bridge, well-balanced features, fair clear skin",
 "ref-30": "a Southeast Asian man in his 20s, attractive and photogenic, oval face, large dark eyes, broad nose, warm brown skin, well-balanced features",
 "ref-31": "a Black man in his 20s, attractive and photogenic, angular face with strong jawline, deep-set eyes, broad nose, rich dark brown skin, well-balanced features",
 "ref-32": "a Middle Eastern man in his 20s, attractive and photogenic, long face with strong jawline, deep-set dark eyes, thick eyebrows, high prominent nose bridge, olive-toned skin, well-balanced features",
},
}

print({k: len(v) for k, v in GROUPS.items()})
print("합계", sum(len(v) for v in GROUPS.values()), "장")

In [ ]:
def run_group(name, overwrite=False):
    """그룹 하나를 생성하고 결과를 보여준다.

    overwrite=False면 이미 있는 파일은 건너뛴다 (재과금 방지).
    """
    imgs = {}
    for rid, desc in GROUPS[name].items():
        path = f"{REF_DIR}/{rid}.png"
        if os.path.exists(path) and not overwrite:
            imgs[rid] = Image.open(path)
            print(f"{rid}  건너뜀 (이미 있음)")
            continue
        t0 = time.time()
        r = client.images.generate(model="gpt-image-2",
                                   prompt=f"{desc}, {SUFFIX}", size="1024x1024", n=1)
        img = Image.open(BytesIO(base64.b64decode(r.data[0].b64_json)))
        img.save(path)
        imgs[rid] = img
        print(f"{rid}  {round(time.time()-t0,1)}초")

    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 6))
    if n == 1: axes = [axes]
    for ax, (rid, im) in zip(axes, imgs.items()):
        ax.imshow(im); ax.set_title(rid, fontsize=13); ax.axis("off")
    plt.suptitle(name, fontsize=15)
    plt.tight_layout(); plt.show()
    return imgs

print("준비 완료")

In [ ]:
res = {}
res["A"] = run_group("A_한국20여")

In [ ]:
res["B"] = run_group("B_한국20남")

In [ ]:
res["C"] = run_group("C_한국30여")

In [ ]:
res["D"] = run_group("D_한국30남")

In [ ]:
res["E"] = run_group("E_한국40")

In [ ]:
res["F"] = run_group("F_한국50")

In [ ]:
res["G"] = run_group("G_외국여")

In [ ]:
res["H"] = run_group("H_외국남")

In [ ]:
import os

files = sorted(f for f in os.listdir(REF_DIR) if f.startswith("ref-"))
total = sum(os.path.getsize(f"{REF_DIR}/{f}") for f in files)
print(f"{len(files)}장  총 {total/1024**2:.1f}MB  평균 {total/len(files)/1024:.0f}KB")